In [1]:
from pydantic import BaseModel, Field
from chromadb import PersistentClient


In [2]:

DB_NAME = "preprocessed_db"  # must match notebook 1 exactly
collection_name = "docs"  # must match notebook 1 exactly

chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_collection(collection_name)

print(f"Loaded collection with {collection.count()} documents")


Loaded collection with 697 documents


In [3]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )


In [4]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv(override=True)

AZURE_ENDPOINT = (
    "https://ankitsinghtheweeknd691-9348-reso.services.ai.azure.com/openai/v1"
)
embedding_model = "qwen--qwen3-embedding-8b"
DEFAULT_MODEL = "mistral-large-2512"

# Client for chat completions (Mistral direct API)
client = OpenAI(
    api_key=os.getenv("MISTRIAL_API_KEY"),
    base_url="https://api.mistral.ai/v1",
)

# Client for embeddings (Azure AI Foundry, hosting qwen)
embedding_client = OpenAI(
    base_url=AZURE_ENDPOINT,
    api_key=os.getenv("AZURE_FOUNDRY_API_KEY"),
)


In [5]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=messages,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "RankOrder",
                "schema": RankOrder.model_json_schema(),
                "strict": True,
            },
        },
    )
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    return [chunks[i - 1] for i in order]


In [6]:
# Inspired by LangChain's Document - let's have something similar


class Result(BaseModel):
    page_content: str
    metadata: dict


In [7]:
RETRIEVAL_K = 20


def fetch_context_unranked(question):
    query = (
        embedding_client.embeddings.create(model=embedding_model, input=[question])
        .data[0]
        .embedding
    )
    
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks


In [8]:
question = "Who won the IIOTY award?"
chunks = fetch_context_unranked(question)


In [9]:
print(chunks)

[Result(page_content="Maxine Thompson Awards and Development Areas\n\nThis chunk concludes Maxine Thompson's HR record with details on her awards, involvement in company initiatives, and areas for future development.\n\n- She was recognized for her contributions with the prestigious Insurellm IIOTY Innovator Award in 2023.  \n- Maxine is currently involved in the women-in-tech initiative and participates in mentorship programs to guide junior employees.  \n- Future development areas include improving her stakeholder communication skills to ensure smoother project transitions and collaboration.", metadata={'source': '../knowledge_base/employees/Maxine Thompson.md', 'type': 'employees'}), Result(page_content="Maxine Thompson Senior Data Engineer Role and Achievements\n\nThis chunk details Maxine Thompson's promotion to Senior Data Engineer in January 2021, her leadership in improving data retrieval times, mentorship role, and recognition as Insurellm Innovator of the Year in 2023.\n\n- *

In [10]:
for chunk in chunks:
    print(chunk.page_content[:15] + "...")


Maxine Thompson...
Maxine Thompson...
Alex Chen 2020 ...
Emily Tran Annu...
Alex Thomson HR...
Emily Carter An...
Alex Chen Caree...
Robert Chen Tec...
Oliver Spencer ...
Emily Carter Ca...
Priya Sharma Ac...
Employer Value ...
Insurellm Produ...
Oliver Spencer ...
Alex Chen Compe...
Alex Thomson Ca...
Maya Thompson C...
Sarah Williams ...
Healthllm Intel...
Analytics, IoT ...


In [11]:
reranked = rerank(question, chunks)


[2, 1, 3, 7, 8, 11, 16, 10, 4, 6, 14, 9, 17, 15, 5, 13, 19, 20, 12, 18]


In [12]:
for chunk in reranked:
    print(chunk.page_content[:15] + "...")


Maxine Thompson...
Maxine Thompson...
Alex Chen 2020 ...
Alex Chen Caree...
Robert Chen Tec...
Priya Sharma Ac...
Alex Thomson Ca...
Emily Carter Ca...
Emily Tran Annu...
Emily Carter An...
Oliver Spencer ...
Oliver Spencer ...
Maya Thompson C...
Alex Chen Compe...
Alex Thomson HR...
Insurellm Produ...
Healthllm Intel...
Analytics, IoT ...
Employer Value ...
Sarah Williams ...


In [13]:
question = "Who went to Manchester University?"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    if "manchester" in c.page_content.lower():
        print(index)


18


In [14]:
reranked = rerank(question, chunks)


[19, 12, 16, 1, 2, 15, 9, 5, 8, 17, 4, 20, 7, 6, 3, 14, 11, 10, 13, 18]


In [15]:
for index, c in enumerate(reranked):
    if "manchester" in c.page_content.lower():
        print(index)


0


In [16]:
reranked[0].page_content


"Jessica Liu Professional Development and Work Style\n\nDescribes Jessica Liu's professional development, work style, and feedback, including her preference for remote work and contributions to open-source projects.\n\n## Other HR Notes\n- **Education:** BS in Computer Science from University of Manchester\n- **Skills:** Proficient in React, TypeScript, HTML/CSS, Jest for testing. Learning Next.js and GraphQL.\n- **Professional Development:** Completed Advanced React Patterns course (2023). Actively contributes to open-source projects.\n- **Work Style:** Prefers remote work. Strong written communicator. Participates actively in team standups and planning sessions.\n- **Feedback:** Reliable developer with good attention to UI details. Improving technical decision-making skills. Would benefit from more ownership of larger features."

In [18]:
def fetch_context_after_rerank(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)


In [19]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""


In [25]:
def make_rag_message(question, history, chunks):
    """
    Builds the full messages list (system + history + new user question)
    given pre-retrieved chunks for a RAG-augmented chat completion call.
    """
    context = "\n\n".join(
        f"# Source: {chunk.metadata.get('source', 'unknown')}\n{chunk.page_content}"
        for chunk in chunks
    )

    system_message = {
        "role": "system",
        "content": SYSTEM_PROMPT.format(context=context),
    }

    messages = [system_message] + history + [{"role": "user", "content": question}]
    return messages


In [26]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=[{"role": "system", "content": message}],
    )
    return response.choices[0].message.content


In [24]:
rewrite_query("Who won the IIOTY award?", [])


'Who won the IIOTY (InsurTech Innovator of the Year) award?'

In [27]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context.
    """
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context_after_rerank(query)
    messages = make_rag_message(question, history, chunks)
    response = client.chat.completions.create(model=DEFAULT_MODEL, messages=messages)
    return response.choices[0].message.content, chunks


In [28]:
answer_question("Who won the IIOTY award?", [])


Who won the IIOTY (Insurtech Innovator of the Year) award?
[1, 7, 2, 10, 14, 9, 13, 18, 20, 19, 12, 3, 8, 11, 17, 4, 6, 5, 15, 16]


('The **Insurellm Innovator of the Year (IIOTY) 2023 award** was won by **Maxine Thompson**, a Senior Data Engineer at Insurellm. She was recognized for her leadership in improving data retrieval times and her contributions to strategic data initiatives.',
 [Result(page_content="Maxine Thompson Senior Data Engineer Role and Achievements\n\nThis chunk details Maxine Thompson's promotion to Senior Data Engineer in January 2021, her leadership in improving data retrieval times, mentorship role, and recognition as Insurellm Innovator of the Year in 2023.\n\n- **January 2021 - Present**: **Senior Data Engineer**  \n  * Maxine was promoted to Senior Data Engineer after successfully leading a pivotal project that improved data retrieval times by 30%. She now mentors junior engineers and is involved in strategic data initiatives, solidifying her position as a valued asset at Insurellm. She was recognized as Insurellm Innovator of the year in 2023, receiving the prestigious IIOTY 2023 award.", 

In [29]:
answer_question("Who went to Manchester University?", [])



Who attended Manchester University?
[3, 1, 4, 2, 6, 8, 9, 11, 15, 17, 20, 12, 7, 13, 14, 5, 16, 10, 18, 19]


('Jessica Liu attended the University of Manchester, where she earned a BS in Computer Science.',
 [Result(page_content="Jessica Liu Professional Development and Work Style\n\nDescribes Jessica Liu's professional development, work style, and feedback, including her preference for remote work and contributions to open-source projects.\n\n## Other HR Notes\n- **Education:** BS in Computer Science from University of Manchester\n- **Skills:** Proficient in React, TypeScript, HTML/CSS, Jest for testing. Learning Next.js and GraphQL.\n- **Professional Development:** Completed Advanced React Patterns course (2023). Actively contributes to open-source projects.\n- **Work Style:** Prefers remote work. Strong written communicator. Participates actively in team standups and planning sessions.\n- **Feedback:** Reliable developer with good attention to UI details. Improving technical decision-making skills. Would benefit from more ownership of larger features.", metadata={'source': '../knowledge_ba